Forward pass (output): $$\sigma(A\sigma(AFW^{(1)})W^{(2)})$$
Loss: $$\mathcal{L}(\sigma(A\sigma(AFW^{(1)})W^{(2)}))$$
$\frac{\partial\mathcal{L}}{W^{(1)}}$: 
$$\frac{\partial\mathcal{L}}{W^{(1)}}=\frac{\partial\mathcal{L}}{\partial X^{(2)}}\frac{\partial X^{(2)}}{\partial S^{(2)}}\frac{\partial S^{(2)}}{\partial X^{(1)}}\frac{\partial X^{(1)}}{\partial S^{(1)}}\frac{\partial S^{(1)}}{\partial W^{(1)}}$$
$$\frac{\partial\mathcal{L}}{W^{(1)}}=F^TA^T(A^T(\mathcal{L}^\prime(X^{(2)})\odot\sigma^{(2)\prime}(S^{(2)}))W^{(2)T}\odot\sigma^{(1)\prime}(S^{(1)}))$$

In [1]:
import torch 
def message_function(A, F, W, b):
    return A @ F @ W + b
def ReLU(X):
    return X * (X > 0) 
def dReLU(X):
    return (X > 0).float()
def sigmoid(X):
    return 1 / (1 + torch.exp(-X))
def dsigmoid(X):
    return sigmoid(X) * (1 - sigmoid(X))
def cross_entropy_loss(y_pred, y_true):
    return -torch.mean(y_true * torch.log(y_pred + 1e-8) + (1 - y_true) * torch.log(1 - y_pred + 1e-8))
def dcross_entropy_loss(y_pred, y_true):
    return -(y_true / (y_pred + 1e-8) - (1 - y_true) / (1 - y_pred + 1e-8)) / y_pred.size(0)
def forward_pass(A, F, W1, W2, b1, b2):
    S1 = message_function(A, F, W1, b1)
    X1 = ReLU(S1)
    S2 = message_function(A, X1, W2, b2)
    X2 = sigmoid(S2) # Output layer with sigmoid activation
    return S1, X1, S2, X2
def backward_pass(A, F, W1, W2, b1, b2, S1, X1, S2, X2, target):
    dL_dX2 = dcross_entropy_loss(X2, target)
    dL_dS2 = dL_dX2 * dsigmoid(S2)
    dL_dW2 = X1.T @ (A.T @ dL_dS2)
    dL_db2 = torch.sum(dL_dS2, dim=0)
    dL_dX1 = A.T @ (dL_dS2 @ W2.T)
    dL_dS1 = dL_dX1 * dReLU(S1)
    dL_dW1 = F.T @ (A.T @ dL_dS1)
    dL_db1 = torch.sum(dL_dS1, dim=0)
    return dL_dW1, dL_db1, dL_dW2, dL_db2

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from torch_geometric.nn import GCNConv
from torch_geometric.transforms import NormalizeFeatures

dataset = Planetoid(root='data/Planetoid', name='Cora', transform=NormalizeFeatures())
data = dataset[0]

print("=== Cora Dataset Statistics ===")
print(f"Number of nodes (papers): {data.num_nodes}")
print(f"Number of edges (citations): {data.num_edges}")
print(f"Number of input features per node: {dataset.num_features}")
print(f"Number of target classes: {dataset.num_classes}")
print(f"Is undirected: {data.is_undirected()}")

class GraphConvolutionalNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GraphConvolutionalNetwork, self).__init__()
        torch.manual_seed(42)
        
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)
        
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x
    
model = GraphConvolutionalNetwork(input_dim=dataset.num_features, hidden_dim=16, output_dim=dataset.num_classes)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()

def train():
    model.train()
    optimizer.zero_grad()
    
    out = model(data.x, data.edge_index)
    
    # Semi-Supervised Learning: Evaluate the loss function only on the node coordinates specified by the train_mask.
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad() # Decorator version of with torch.no_grad():
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    predictions = out.argmax(dim=1)
    
    accuracies = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        correct = (predictions[mask] == data.y[mask]).sum()
        accuracy = int(correct) / int(mask.sum())
        accuracies.append(accuracy)
        
    return accuracies

def plot_embeddings(embeddings, labels, epoch, title):
    # Use t-SNE to reduce the n-dimensional hidden state down to 2 dimensions (X and Y)
    tsne = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto')
    embeddings_2d = tsne.fit_transform(embeddings.detach().cpu().numpy())
    
    # The class names for the legend
    class_names = [
        "Case-Based", "Genetic Algs", "Neural Networks", 
        "Probabilistic", "Reinforcement", "Rule Learning", "Theory"
    ]
    
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(
        embeddings_2d[:, 0], 
        embeddings_2d[:, 1], 
        c=labels.cpu().numpy(), 
        cmap='Set1', 
        s=20, 
        alpha=0.8
    )
    
    plt.legend(handles=scatter.legend_elements()[0], labels=class_names, title="Classes")
    plt.title(f"{title} (Epoch {epoch})")
    plt.axis('off')
    plt.show()

print("\n--- Beginning Training Loop ---")
for epoch in range(1, 201):
    loss_val = train()
    
    if epoch % 20 == 0 or epoch == 1:
        train_acc, val_acc, test_acc = test()
        print(f"Epoch: {epoch:3d} | Loss: {loss_val:.4f} | Train Acc: {train_acc*100:.1f}% | Val Acc: {val_acc*100:.1f}% | Test Acc: {test_acc*100:.1f}%")

_, _, final_test_acc = test()
print(f"\nFinal Generalization Accuracy on Unseen Test Nodes: {final_test_acc*100:.2f}%")

model.eval()
final_embeddings = model(data.x, data.edge_index)
print("Plotting trained embeddings...")
plot_embeddings(final_embeddings, data.y, 200, "GNN Embeddings AFTER Training")

=== Cora Dataset Statistics ===
Number of nodes (papers): 2708
Number of edges (citations): 10556
Number of input features per node: 1433
Number of target classes: 7
Is undirected: True

--- Beginning Training Loop ---
Epoch:   1 | Loss: 1.9459 | Train Acc: 32.1% | Val Acc: 14.4% | Test Acc: 15.7%
Epoch:  20 | Loss: 1.7174 | Train Acc: 85.7% | Val Acc: 52.6% | Test Acc: 53.6%
Epoch:  40 | Loss: 1.3281 | Train Acc: 96.4% | Val Acc: 70.4% | Test Acc: 70.3%
Epoch:  60 | Loss: 0.9540 | Train Acc: 99.3% | Val Acc: 76.0% | Test Acc: 77.2%
Epoch:  80 | Loss: 0.6925 | Train Acc: 99.3% | Val Acc: 77.6% | Test Acc: 79.0%
Epoch: 100 | Loss: 0.5591 | Train Acc: 99.3% | Val Acc: 76.6% | Test Acc: 79.5%
Epoch: 120 | Loss: 0.4700 | Train Acc: 99.3% | Val Acc: 77.4% | Test Acc: 80.2%
Epoch: 140 | Loss: 0.4071 | Train Acc: 99.3% | Val Acc: 77.8% | Test Acc: 80.1%
Epoch: 160 | Loss: 0.3849 | Train Acc: 100.0% | Val Acc: 77.8% | Test Acc: 80.1%
Epoch: 180 | Loss: 0.3435 | Train Acc: 100.0% | Val Acc: 77.

ValueError: too many values to unpack (expected 2)